##### ARTI 560 - Computer Vision

## Action Recognition - Exercise

### Objective

In this exercise, you will train a deep learning model to recognize three specific human actions using the [UCF11 (YouTube Action) dataset](https://www.crcv.ucf.edu/data/UCF_YouTube_Action.php) and validate the model's real-world performance using external video data.

*[Note: This notebook is based on [this](https://github.com/Sumaya2026/learnopencv/tree/master/Optical-Flow-Estimation-using-Deep-Learning-RAFT) GitHub Repository by LearnOpenCV]*


#### Tasks

- Choose **three classes** from the UCF11 dataset (e.g., Basketball Shooting, Biking, Tennis Swinging, etc.).
- Preprocess the dataset.
- Split the data into training and testing.
- Create and train the model.
- Save the trained model .
    **Important Note**: The final trained model must be saved with a filename that includes your name. This is a mandatory step for the submission.
    ```
    # Example Code
    student_name = "YourName" # Replace with your actual name
    save_path = f"{student_name}_ucf11_model.h5"
    model.save(save_path)
    print(f"Model saved as {save_path}")
    ```
- Validate the model on 3 Youtube videos, each clearly showing one of your three chosen action classes.


In [1]:
!wget -nc --no-check-certificate https://www.crcv.ucf.edu/data/UCF11_updated_mpg.rar

File ‘UCF11_updated_mpg.rar’ already there; not retrieving.



In [19]:
!rm -rf UCF11_updated_mpg
!unrar x -o+ UCF11_updated_mpg.rar


UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from UCF11_updated_mpg.rar

Creating    UCF11_updated_mpg                                         OK
Creating    UCF11_updated_mpg/basketball                              OK
Creating    UCF11_updated_mpg/basketball/v_shooting_01                OK
Extracting  UCF11_updated_mpg/basketball/v_shooting_01/v_shooting_01_01.mpg       0%  OK 
Extracting  UCF11_updated_mpg/basketball/v_shooting_01/v_shooting_01_02.mpg       0%  OK 
Extracting  UCF11_updated_mpg/basketball/v_shooting_01/v_shooting_01_03.mpg       0%  OK 
Extracting  UCF11_updated_mpg/basketball/v_shooting_01/v_shooting_01_04.mpg       0%  OK 
Extracting  UCF11_updated_mpg/basketball/v_shooting_01/v_shooting_01_05.mpg       0%  OK 
Extracting  UCF11_updated_mpg/basketball/v_shooting_01/v_shooting_01_06.mpg       0%  OK 
Extracting  UCF11_updated_mpg/basketball/v_shooting_01/v_shooting_01_07.

In [20]:
import os
print(os.listdir("UCF11_updated_mpg")[:10])

['horse_riding', 'swing', 'basketball', 'trampoline_jumping', 'golf_swing', 'soccer_juggling', 'volleyball_spiking', 'biking', 'walking', 'tennis_swing']


In [21]:
classes_list = ["biking", "walking", "horse_riding"]
dataset_directory = "UCF11_updated_mpg"

image_height, image_width = 64, 64
max_images_per_class = 8000

In [22]:
import os
import cv2
import random
import numpy as np
import datetime as dt
import tensorflow as tf
import matplotlib.pyplot as plt
%matplotlib inline

from collections import deque
from sklearn.model_selection import train_test_split

from tensorflow.keras.layers import *
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, GlobalAveragePooling2D, Dense


In [23]:
def frames_extraction(video_path):
    frames_list = []

    video_reader = cv2.VideoCapture(video_path)

    while True:
        success, frame = video_reader.read()

        if not success:
            break

        resized_frame = cv2.resize(frame, (image_height, image_width))
        normalized_frame = resized_frame / 255
        frames_list.append(normalized_frame)

    video_reader.release()

    return frames_list

In [24]:
def create_dataset():
    features = []
    labels = []

    for class_index, class_name in enumerate(classes_list):
        print(f"Extracting Data of Class: {class_name}")

        temp_features = []   # IMPORTANT: inside the loop

        class_path = os.path.join(dataset_directory, class_name)

        for root, dirs, files in os.walk(class_path):
            for file_name in files:
                if file_name.endswith((".avi", ".mpg", ".mp4")):
                    video_file_path = os.path.join(root, file_name)
                    frames = frames_extraction(video_file_path)
                    temp_features.extend(frames)

        selected_frames = random.sample(
            temp_features,
            min(max_images_per_class, len(temp_features))
        )

        features.extend(selected_frames)
        labels.extend([class_index] * len(selected_frames))

    features = np.asarray(features)
    labels = np.array(labels)

    return features, labels

In [25]:
features, labels = create_dataset()

print("Features shape:", features.shape)
print("Labels shape:", labels.shape)

Extracting Data of Class: biking
Extracting Data of Class: walking
Extracting Data of Class: horse_riding
Features shape: (24000, 64, 64, 3)
Labels shape: (24000,)


In [26]:
seed_constant = 23
model_output_size = 3


one_hot_encoded_labels = to_categorical(labels, num_classes=3)

print("Classes:", classes_list)
print("Labels unique:", np.unique(labels))
print("One-hot shape:", one_hot_encoded_labels.shape)

features_train, features_test, labels_train, labels_test = train_test_split(
    features,
    one_hot_encoded_labels,
    test_size=0.2,
    shuffle=True,
    random_state=seed_constant
)

Classes: ['biking', 'walking', 'horse_riding']
Labels unique: [0 1 2]
One-hot shape: (24000, 3)


In [27]:
def create_model():
    model = Sequential()

    model.add(Conv2D(64, (3,3), activation='relu',
                     input_shape=(image_height, image_width, 3)))
    model.add(Conv2D(64, (3,3), activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2,2)))
    model.add(GlobalAveragePooling2D())

    model.add(Dense(256, activation='relu'))
    model.add(BatchNormalization())

    model.add(Dense(model_output_size, activation='softmax'))

    return model

model = create_model()
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 62, 62, 64)     │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 60, 60, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 60, 60, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │           771 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 57,411 (224.26 KB)

 Trainable params: 56,771 (221.76 KB)

 Non-trainable params: 640 (2.50 KB)

In [28]:
model.compile(
    loss='categorical_crossentropy',
    optimizer='Adam',
    metrics=['accuracy']
)

In [29]:
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    features_train,
    labels_train,
    epochs=15,
    batch_size=16,
    validation_split=0.2,
    callbacks=[early_stopping]
)

Epoch 1/15
960/960 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.6923 - loss: 0.7088 - val_accuracy: 0.7471 - val_loss: 0.6070
Epoch 2/15
960/960 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8331 - loss: 0.4266 - val_accuracy: 0.6135 - val_loss: 1.9754
Epoch 3/15
960/960 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8859 - loss: 0.3033 - val_accuracy: 0.7388 - val_loss: 0.9083
Epoch 4/15
960/960 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9156 - loss: 0.2296 - val_accuracy: 0.8596 - val_loss: 0.3884
Epoch 5/15
960/960 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9322 - loss: 0.1906 - val_accuracy: 0.9146 - val_loss: 0.2499
Epoch 6/15
960/960 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9385 - loss: 0.1695 - val_accuracy: 0.8977 - val_loss: 0.2784
Epoch 7/15
960/960 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9509 - loss: 0.1387 - val_accuracy: 0.8979 - val_loss: 0.3215
Epoch 8/15
960/960 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9604 - loss: 0.1131 - val_accuracy: 0.

In [30]:
loss, accuracy = model.evaluate(features_test, labels_test)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

150/150 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9194 - loss: 0.2390
Test Loss: 0.23895163834095
Test Accuracy: 0.9193750023841858


In [31]:
model_name = "Mozoon_ucf11_model.h5"
model.save(model_name)

print("Saved:", model_name)

Saved: Mozoon_ucf11_model.h5


In [32]:
!pip install -q yt-dlp

In [35]:
youtube_links = {
    "walking": "https://youtu.be/6MEunPE_HAM?si=hRzma1P32RGan7Cx",
    "biking": "https://youtu.be/0MLnC3bzXgQ?si=-4xFuiQgCV_kZEy6",
    "horse_riding": "https://youtu.be/gbF0gfuf8rQ?si=HUWJK8NPilyZEnNp"
}

import os

os.makedirs("youtube_videos", exist_ok=True)

for label, link in youtube_links.items():
    output_path = f"youtube_videos/{label}.mp4"
    !yt-dlp -f mp4 -o "{output_path}" "{link}"

         Pre-merged mp4 formats are not available from all sites, or may only be available in lower quality.
         To prioritize the best h264 video and aac audio in an mp4 container, use "-t mp4" instead.
         If you know what you are doing and want a pre-merged mp4 format, use "-f b[ext=mp4]" instead to suppress this warning
[youtube] Extracting URL: https://youtu.be/6MEunPE_HAM?si=hRzma1P32RGan7Cx
[youtube] 6MEunPE_HAM: Downloading webpage
[youtube] 6MEunPE_HAM: Downloading android vr player API JSON
[info] 6MEunPE_HAM: Downloading 1 format(s): 18
[download] youtube_videos/walking.mp4 has already been downloaded
[download] 100% of    2.58MiB
         Pre-merged mp4 formats are not available from all sites, or may only be available in lower quality.
         To prioritize the best h264 video and aac audio in an mp4 container, use "-t mp4" instead.
         If you know what you are doing and want a pre-merged mp4 format, use "-f b[ext=mp4]" instead to suppress this warning
[you

In [36]:
def predict_video(video_path):
    frames = frames_extraction(video_path)

    if len(frames) == 0:
        print("No frames extracted.")
        return

    # focus on middle part
    start = int(len(frames) * 0.2)
    end = int(len(frames) * 0.8)
    frames = frames[start:end]

    MAX_FRAMES = 1000

    if len(frames) > MAX_FRAMES:
        step = len(frames) // MAX_FRAMES
        frames = frames[::step]

    predictions = model.predict(np.array(frames), verbose=0)
    avg_pred = np.mean(predictions, axis=0)

    predicted_class = classes_list[np.argmax(avg_pred)]
    confidence = np.max(avg_pred)

    print("Frames used:", len(frames))
    print("Predicted:", predicted_class)
    print("Confidence:", confidence)

In [37]:
for label in youtube_links.keys():
    video_path = f"youtube_videos/{label}.mp4"

    print("Expected Action:", label)
    predict_video(video_path)
    print("-" * 40)

Expected Action: walking
Frames used: 924
Predicted: biking
Confidence: 0.91601014
----------------------------------------
Expected Action: biking
Frames used: 1283
Predicted: biking
Confidence: 0.56614006
----------------------------------------
Expected Action: horse_riding
Frames used: 1238
Predicted: horse_riding
Confidence: 0.4521531
----------------------------------------
